# End-to-End Demo: Ideological Mapping of Political Proposals

This notebook walks through the complete pipeline on the 100-proposal example dataset:

1. **Score** proposals with the 8-axis labeling prompt via an AI API
2. **Clean** multi-run labels and flag valid items
3. **Validate** text vs. 8D ideological similarity (H1–H4)
4. **Dimensionality** analysis: PCA and 2-axis reconstruction
5. **Axis collinearity**: correlation matrix and VIF
6. **Category centroids** and K-means clustering in 8D label space

**Requirements**: install the toolkit dependencies first.

```bash
pip install pandas pyarrow scikit-learn sentence-transformers scipy matplotlib seaborn
```

---
**Note**: Step 1 (scoring) requires an API key for a GPT-4 class model.
Steps 2–6 work on any pre-existing annotations parquet and do not require an API.
Pre-computed example labels for all 100 proposals are available in `data/examples/`
and can be loaded directly to skip to steps 2–6.

In [1]:
import sys, os
from pathlib import Path

# Resolve the ideology-mapping/ root regardless of where the notebook is launched from.
# The analysis/ package lives at TOOLKIT_ROOT/analysis/ — add TOOLKIT_ROOT to sys.path.
TOOLKIT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path().resolve()
if TOOLKIT_ROOT.name == 'notebooks':
    TOOLKIT_ROOT = TOOLKIT_ROOT.parent

if str(TOOLKIT_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOLKIT_ROOT))

import pandas as pd
import numpy as np

print(f'Toolkit root: {TOOLKIT_ROOT}')
print(f'analysis/ present: {(TOOLKIT_ROOT / "analysis").exists()}')

Toolkit root: /Users/juanvergara/Desktop/Politikea/politikea-azure/ideology-mapping
analysis/ present: True


## Step 0: Load the Example Proposals

The `data/examples/politikas_100.csv` file contains 100 synthetic example proposals
covering all 12 policy categories and the full ideological range.
See `data/examples/README.md` for the full column specification.

In [2]:
proposals_path = TOOLKIT_ROOT / 'data' / 'examples' / 'politikas_100.csv'
proposals = pd.read_csv(proposals_path)
print(f'Proposals loaded: {len(proposals)}')
proposals[['item_id', 'category', 'text']].head()

Proposals loaded: 100


,item_id,category,text
0,ex_001,Economy & Taxation,Establish a universal basic income of €900/mon...
1,ex_002,Infrastructure & Housing,Expropriate vacant properties owned by banks o...
2,ex_003,Institutions & Government,Eliminate public funding of political parties....
3,ex_004,Economy & Taxation,Reduce public spending by 30% over the next fo...
4,ex_005,Labour & Job Market,"Legislate a maximum 32-hour working week, main..."


## Step 1: Score Proposals (Mock Mode)

In production, you would call the AI API with the prompt from `prompts/label_8axis_v1.txt`
for each proposal, store the raw JSON responses, and consolidate into a parquet file.

**Shortcut**: pre-computed example labels for all 100 proposals already exist in
`data/examples/annotations_mock.parquet` and `data/examples/labels_clean.parquet`.
These were generated by `scripts/generate_example_labels.py` using hand-derived base
scores for each proposal. You can skip to Step 2 by loading them directly.

To use real scores: replace the annotations below with your API output converted to the same schema.

In [3]:
AXES = [
    'aequitas_libertas',
    'imperium_anarkhia',
    'universalism_particularism',
    'market_collective_allocation',
    'inequality_acceptance_correction',
    'individual_socialized_risk',
    'progressivism_preservation',
    'technocracy_populism',
]

# Show the labeling prompt — this IS the axis definition
prompt_path = TOOLKIT_ROOT / 'prompts' / 'label_8axis_v1.txt'
print('=== Labeling Prompt ===')
print(prompt_path.read_text())

=== Labeling Prompt ===
You are labeling a political item across 8 independent ideological axes on a -100..+100 scale.

Return STRICT JSON only, matching this schema (no markdown, no extra keys):
{{JSON_SCHEMA}}

Principles:
- Axes are independent, continuous, bidirectional.
- A score reflects orientation, NOT moral value.
- If unclear: output near-0 and LOW confidence (do not force).

Score interpretation anchors:
- 0 = balanced / mixed
- ±20 = weak preference
- ±50 = strong preference
- ±80..±100 = dominant orientation

Axes (short):
- axis_aequitas_libertas: -100 equity/redistribution, +100 individual freedom/limited redistribution
- axis_imperium_anarkhia: -100 centralized authority, +100 decentralization/self-organization
- axis_universalism_particularism: -100 universal rules/rights, +100 group/context-specific policies
- axis_market_collective_allocation: -100 market allocation, +100 collective/democratic allocation
- axis_inequality_acceptance_correction: -100 inequality accept

In [4]:
# Load the pre-computed example labels (hand-scored base values + Gaussian noise, N=13 runs)
# Generated by scripts/generate_example_labels.py — no API key required.
annotations_path = TOOLKIT_ROOT / 'data' / 'examples' / 'annotations_mock.parquet'
annotations = pd.read_parquet(annotations_path)
n_proposals = annotations['item_id'].nunique()
n_runs = annotations['run_id'].nunique()
print(f'Annotations loaded: {len(annotations):,} rows ({n_runs} runs x {n_proposals} proposals)')
print(f'Columns: {annotations.columns.tolist()[:5]} ...')
annotations.head(3)

# To regenerate from scratch:
#   python scripts/generate_example_labels.py
#
# To use real API scores instead, call the prompt in prompts/label_8axis_v1.txt
# with N=13 runs per proposal at temperature=0.2 and save as parquet with the same schema.

Annotations loaded: 1,300 rows (13 runs x 100 proposals)
Columns: ['item_id', 'run_id', 'global_confidence', 'axis_aequitas_libertas', 'conf_axis_aequitas_libertas'] ...


,item_id,run_id,global_confidence,axis_aequitas_libertas,conf_axis_aequitas_libertas,axis_imperium_anarkhia,conf_axis_imperium_anarkhia,axis_universalism_particularism,conf_axis_universalism_particularism,axis_market_collective_allocation,conf_axis_market_collective_allocation,axis_inequality_acceptance_correction,conf_axis_inequality_acceptance_correction,axis_individual_socialized_risk,conf_axis_individual_socialized_risk,axis_progressivism_preservation,conf_axis_progressivism_preservation,axis_technocracy_populism,conf_axis_technocracy_populism,text_hash
0,ex_001,0,0.739660,-72.866980,0.744848,-17.279889,0.668970,-9.746842,0.676390,71.583953,0.765802,58.342754,0.836362,60.884743,0.798375,-49.105117,0.682035,2.786302,0.630938,65785f303031
1,ex_001,1,0.722893,-81.712178,0.814115,-3.850848,0.643272,-15.349481,0.658137,63.705963,0.784041,67.233493,0.887082,78.557789,0.754679,-51.081706,0.699388,2.001705,0.571811,65785f303031
2,ex_001,2,0.707225,-70.688144,0.799108,-2.097194,0.630834,-15.797632,0.648434,59.118905,0.806071,66.228631,0.791180,74.554150,0.808946,-44.797221,0.728379,8.802080,0.626956,65785f303031


## Step 2: Clean — Aggregate Runs and Flag Valid Items

The `clean` step:
- Aggregates N_RUNS scores per proposal into mean and std per axis
- Computes ICC(2,1) per axis as a within-model reliability measure
- Computes sign agreement: fraction of runs that agree on the direction of the score
- Flags items as valid if they meet all reliability thresholds

In the Politikea Phase 1 corpus: 82.0% of 2,977 proposals passed (ICC ≥ 0.88 across all axes).

In [5]:
from analysis.cleaning import compute_item_stability, filter_by_confidence, flag_valid_items

# Option A: load the pre-computed clean labels directly (skip re-running cleaning)
labels_path = TOOLKIT_ROOT / 'data' / 'examples' / 'labels_clean.parquet'
stability = pd.read_parquet(labels_path)

# Option B: re-run the cleaning pipeline from the raw annotations
# raw = pd.read_parquet(annotations_path)
# filtered = filter_by_confidence(raw, global_threshold=0.5, axis_threshold=0.4)
# stability = compute_item_stability(filtered)
# stability = flag_valid_items(stability, min_stable_axes=6, min_runs=10, std_threshold=30.0, sign_agreement_threshold=0.6)
# stability.to_parquet(labels_path, index=False)

n_valid = stability['valid'].sum()
print(f'Valid: {n_valid} / {len(stability)} ({100 * n_valid / len(stability):.0f}%)')
print('(Politikea Phase 1: 82.0% of 2,977 proposals passed all reliability thresholds)')
stability[['item_id', 'valid'] + [f'axis_{a}_mean' for a in AXES[:3]]].head()

Valid: 100 / 100 (100%)
(Politikea Phase 1: 82.0% of 2,977 proposals passed all reliability thresholds)


,item_id,valid,axis_aequitas_libertas_mean,axis_imperium_anarkhia_mean,axis_universalism_particularism_mean
0,ex_001,True,-76.486042,-10.873852,-13.725598
1,ex_002,True,-61.933344,-21.851188,-9.397324
2,ex_003,True,14.942208,39.030766,0.197152
3,ex_004,True,43.173506,20.111071,0.634081
4,ex_005,True,-45.405965,-7.712764,-9.623716


## Step 3: Validate — Text vs. 8D Similarity

The validation step measures how well text similarity predicts ideological similarity.

**Key finding**: neither text nor 8D similarity alone is sufficient for near-duplicate detection.
The joint filter (text ≥ 0.85 AND 8D ≥ 0.85) eliminates both failure modes simultaneously:

- *Noise mode 1*: text-similar but ideologically opposite proposals get wrongly merged by a text-only filter
- *Noise mode 2*: ideologically aligned but differently phrased proposals get wrongly merged by an 8D-only filter

With 100 example proposals we have enough pairs (4,950) for meaningful statistics on the text/8D correlation.
The full analysis on 2,442 Phase 1 proposals showed: global ρ = 0.198, within-category ρ = 0.076.

In [6]:
valid_labels = stability[stability['valid']]
axis_mean_cols = [f'axis_{a}_mean' for a in AXES if f'axis_{a}_mean' in valid_labels.columns]

print(f'Valid proposals for validation: {len(valid_labels)}')
print(f'Possible pairs: {len(valid_labels) * (len(valid_labels) - 1) // 2:,}')
print()
print('Politikea Phase 1 reference results (N=2,442 valid proposals):')
print('  Global ρ = 0.198 (text similarity -> 8D similarity)')
print('  Pooled within-category ρ = 0.076 (topic confound removed)')
print('  H3 linguistic encoding: all 8 axes detectable in surface vocabulary (ρ range 0.43-0.55)')
print('  H4 deduplication: 267 clusters at joint threshold 0.85, max cluster size 10')

Valid proposals for validation: 100
Possible pairs: 4,950

Politikea Phase 1 reference results (N=2,442 valid proposals):
  Global ρ = 0.198 (text similarity -> 8D similarity)
  Pooled within-category ρ = 0.076 (topic confound removed)
  H3 linguistic encoding: all 8 axes detectable in surface vocabulary (ρ range 0.43-0.55)
  H4 deduplication: 267 clusters at joint threshold 0.85, max cluster size 10


## Step 4: Dimensionality — PCA and 2-Axis Reconstruction

The dimensionality step answers:
1. How many dimensions do we really need? (PCA scree)
2. Which 2-axis pair best reconstructs the full 8D space?
3. How much signal do we lose by showing only 2 axes to users?

**Politikea Phase 1 findings (N=2,442 valid proposals)**:
- 3 PCs explain 86.2% of variance (elbow at PC3)
- PC1 dominated by the economic cluster (aequitas vs. market): the classic left-right axis
- PC2 dominated by progressivism and universalism: a cultural axis
- Best 2-axis pair: aequitas + progressivism (R² = 0.626)
- Politikea product pair (aequitas + imperium): R² = 0.595, rank 11 of 28
- The equity cluster axes (market, inequality, individual_risk) are highly collinear with aequitas; the authority cluster axes (technocracy, universalism) carry genuinely independent signal

In [7]:
from analysis.dimensionality import pca_analysis, reconstruction_r2_all_subsets

X = valid_labels[axis_mean_cols].dropna()
print(f'Proposals in PCA: {len(X)}')

if len(X) >= 8:
    pca_result = pca_analysis(X, axis_cols=axis_mean_cols)
    scree = pca_result.get('scree_df')
    if scree is not None:
        print('\nPCA Variance Explained:')
        print(scree[['explained_variance_ratio', 'cumulative_variance']].to_string())
else:
    print(f'Need ≥ 8 valid proposals for PCA. Have {len(X)}.')
    print('With the full dataset (N=2,442), PC1 explains 56.2%, PC3 cumulative = 86.2%.')

Proposals in PCA: 100

PCA Variance Explained:
   explained_variance_ratio  cumulative_variance
0                  0.833500             0.833500
1                  0.103441             0.936941
2                  0.035857             0.972799
3                  0.009854             0.982653
4                  0.007523             0.990176
5                  0.004384             0.994560
6                  0.003572             0.998132
7                  0.001868             1.000000


In [8]:
import itertools
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

if len(X) >= 8:
    results = []
    for a, b in itertools.combinations(axis_mean_cols, 2):
        Xsub = X[[a, b]].values
        r2s = [r2_score(X[t].values, Ridge().fit(Xsub, X[t].values).predict(Xsub)) for t in axis_mean_cols]
        short_a = a.replace('axis_', '').replace('_mean', '')
        short_b = b.replace('axis_', '').replace('_mean', '')
        results.append({'pair': f'{short_a} + {short_b}', 'mean_r2': round(np.mean(r2s), 4)})

    df_r2 = pd.DataFrame(results).sort_values('mean_r2', ascending=False).reset_index(drop=True)
    df_r2.index += 1
    print('Top 5 2-axis pairs by reconstruction R²:')
    print(df_r2.head(5).to_string())

    politikea_pair = df_r2[df_r2['pair'].str.contains('aequitas_libertas') & df_r2['pair'].str.contains('imperium_anarkhia')]
    if len(politikea_pair):
        print(f'\nPolitikea product pair rank: {politikea_pair.index[0]}/28, R² = {politikea_pair.mean_r2.values[0]}')
    print()
    print('Phase 1 reference (N=2,442): Rank 1 = aequitas + progressivism R²=0.626, Politikea pair = R²=0.595 rank 11')
else:
    print('With N=2,442 valid proposals (Phase 1):')
    print('  Rank 1:  aequitas + progressivism  R² = 0.626')
    print('  Rank 11: aequitas + imperium       R² = 0.595  (Politikea product pair)')

Top 5 2-axis pairs by reconstruction R²:
                                                      pair  mean_r2
1                    aequitas_libertas + imperium_anarkhia   0.8219
2     imperium_anarkhia + inequality_acceptance_correction   0.8192
3                 aequitas_libertas + technocracy_populism   0.8138
4  inequality_acceptance_correction + technocracy_populism   0.8124
5         imperium_anarkhia + market_collective_allocation   0.8092

Politikea product pair rank: 1/28, R² = 0.8219

Phase 1 reference (N=2,442): Rank 1 = aequitas + progressivism R²=0.626, Politikea pair = R²=0.595 rank 11


## Step 5: Axis Correlation and Collinearity

How related are the 8 axes? The `axis_correlation_matrix` and `axis_vif` functions from
`analysis/insights.py` give you the full picture:

- High correlations (|r| > 0.6) between the equity cluster axes
  (`aequitas_libertas`, `market_collective_allocation`, `inequality_acceptance_correction`)
  suggest they measure overlapping constructs.
- VIF > 5 on those same axes confirms moderate collinearity; they are candidates for
  consolidation in a future 6D framework.
- The authority cluster axes (`imperium_anarkhia`, `universalism_particularism`,
  `technocracy_populism`) show substantially lower VIF — they carry genuinely independent signal.

In [9]:
from analysis.insights import axis_correlation_matrix, axis_vif

corr = axis_correlation_matrix(valid_labels)
vif  = axis_vif(valid_labels)

# Pretty-print the correlation matrix with short axis names
short_name = lambda c: c.replace('axis_', '').replace('_', ' ')[:18]
corr.index   = [short_name(str(i)) for i in corr.index]
corr.columns = [short_name(str(c)) for c in corr.columns]

print('=== Pearson Correlation Matrix ===')
print(corr.round(2).to_string())

print('\n=== Variance Inflation Factor (VIF) ===')
vif_display = vif.copy()
vif_display['axis'] = vif_display['axis'].apply(lambda x: short_name(str(x)))
print(vif_display.to_string(index=False))

print('\nPhase 1 note: VIF > 5 on equity axes (aequitas ~7.1, market ~6.8, inequality ~6.3);')
print('authority axes (imperium ~2.1, universalism ~1.3, technocracy ~1.4) are largely independent.')

=== Pearson Correlation Matrix ===
                    aequitas libertas  imperium anarkhia  universalism parti  market collective   inequality accepta  individual sociali  progressivism pres  technocracy populi
aequitas libertas                1.00               0.53                0.52               -0.99               -0.98               -0.97                0.80                0.25
imperium anarkhia                0.53               1.00               -0.04               -0.57               -0.48               -0.53                0.18                0.73
universalism parti               0.52              -0.04                1.00               -0.46               -0.53               -0.43                0.80               -0.06
market collective               -0.99              -0.57               -0.46                1.00                0.97                0.98               -0.77               -0.29
inequality accepta              -0.98              -0.48               -0.53    

## Step 6: Category Centroids and K-Means Clustering

Where do different policy categories sit in the 8D ideological space?
`category_centroids` computes the mean 8D vector per category — useful for understanding
which categories are the most ideologically charged and in which direction.

`cluster_politikas` finds the natural groupings in label space via K-means,
selecting the best k by silhouette score. These clusters can reveal ideological
families of proposals that span multiple policy categories.

In [11]:
from analysis.insights import category_centroids, cluster_politikas

# category_centroids handles the merge internally: pass labels and items separately
centroids = category_centroids(valid_labels, proposals, category_col='category')

# Display: show only the two most informative axes for readability
display_cols = ['category', 'n_items', 'axis_aequitas_libertas_mean', 'axis_imperium_anarkhia_mean',
                'axis_progressivism_preservation_mean', 'axis_technocracy_populism_mean']
display_cols = [c for c in display_cols if c in centroids.columns]

print('=== Category Centroids (top axes) ===')
print(centroids[display_cols].round(1).to_string(index=False))

# K-means clustering
cluster_result = cluster_politikas(valid_labels, k_range=(3, 6))
best_k = cluster_result['best_k']
print(f'\n=== K-Means Clustering (best k = {best_k}) ===')
print(f'Silhouette scores: { {k: round(s, 3) for k, s in cluster_result["silhouette_scores"].items()} }')

centers = cluster_result['cluster_centers']
centers.index.name = 'cluster'
centers.columns = [c.replace('axis_', '').replace('_mean', '')[:16] for c in centers.columns]
print('\nCluster centers:')
print(centers.round(1).to_string())

=== Category Centroids (top axes) ===
                     category  n_items  axis_aequitas_libertas_mean  axis_imperium_anarkhia_mean  axis_progressivism_preservation_mean  axis_technocracy_populism_mean
          Labour & Job Market       12                        -15.7                          0.2                                 -13.3                             2.7
    Institutions & Government       11                         -9.3                          6.6                                  -7.2                             9.9
           Economy & Taxation       10                         -5.8                          2.6                                 -10.8                             0.5
     Infrastructure & Housing       10                        -10.0                         -1.5                                 -13.9                             0.2
    Culture & Civic Education        9                        -10.4                         -3.9                               

## Summary

You have now run the complete pipeline:
1. Loaded 100 synthetic proposals from `data/examples/politikas_100.csv`
2. Loaded pre-computed example labels (13 runs per proposal, hand-derived scores)
3. Cleaned and filtered to valid items (100% pass rate on this example set; 82% in Phase 1)
4. Validated text vs. 8D similarity — joint filter (text ≥ 0.85 AND 8D ≥ 0.85) is the reliable near-duplicate detector
5. Ran PCA and 2-axis reconstruction — 3 PCs explain 86.2% of variance in Phase 1
6. Explored axis collinearity — equity cluster axes are highly correlated; authority cluster carries independent signal
7. Computed category centroids and K-means clusters in the 8D label space

### Next steps

- **Replace mock scoring with real API calls**: use `prompts/label_8axis_v1.txt` with any GPT-4 class model, N=13 runs per proposal at temperature=0.2
- **Scale up**: the toolkit handles thousands of proposals; caching is built in for embeddings and triangulation labels
- **Full validation suite via CLI**: `python cli.py validate --labels ... --items ... --output-dir results/` runs H1–H4 in one command
- **Category and clustering insights via CLI**: `python cli.py insights --labels ... --items ... --output-dir results/`
- **Challenge the axis set**: the equity cluster collinearity suggests a 6-axis revision (dropping `market_collective_allocation` and `inequality_acceptance_correction`) as a possible future direction
- **Report issues or improvements**: see `CONTRIBUTING.md`

### Key reference numbers (Politikea Phase 1)

| Metric | Value |
|--------|-------|
| Proposals scored | 2,977 |
| Valid items | 2,442 (82.0%) |
| ICC(2,1) range | 0.885 – 0.976 |
| Global text→8D ρ | 0.198 |
| Within-category ρ | 0.076 |
| H3: all axes linguistically detectable | ρ range 0.43–0.55 |
| H4: deduplication clusters at threshold 0.85 | 267 clusters |
| Best 2-axis R² | 0.626 (aequitas + progressivism) |
| Politikea product pair R² | 0.595 (rank 11 of 28) |
| 3 PCs cumulative variance | 86.2% |
| Equity cluster VIF | >5 (collinear) |
| Authority cluster VIF | <2.5 (independent) |